In [21]:
import pandas as pd
df = pd.read_csv('/Volumes/sd/faith/MTCSB/projects/P4-barcoding_strains/20251104-analysis/clustered/P4C1T8/c5_ST1_clustered.csv')
ntc_cols = [col for col in df.columns if 'NTC' in col or 'WATER' in col.upper()]

df_stack = df.set_index('umi_seq')[ntc_cols].stack().reset_index()
df_stack.columns = ['umi_seq', 'sample', 'clustered_molecules']
noise_threshold = df_stack['clustered_molecules'].quantile(0.99)

df_filtered = df.copy()
numeric_cols = df_filtered.select_dtypes(include=['int64', 'float64']).columns
df_filtered[numeric_cols] = df_filtered[numeric_cols].where(df_filtered[numeric_cols] >= noise_threshold, 0)

#drop everything where gavage.lower() is 0
gavage_cols = [col for col in df.columns if 'gavage' in col.lower()]
df_filtered = df_filtered[df_filtered[gavage_cols].sum(axis=1) > 0]

df_filtered.to_csv('/Volumes/sd/faith/MTCSB/projects/P4-barcoding_strains/20251104-analysis/clustered/P4C1T8/c5_ST1_clustered_denoised.csv',index = False )

In [9]:
df_stack = df.set_index('umi_seq').stack().reset_index()
df_stack.columns = ['umi_seq', 'sample', 'clustered_molecules']
df_stack

,umi_seq,sample,clustered_molecules
0,ST1-AAAAGACTGGGCCTTTCG,s9_gavage-DNA,14340
1,ST1-AAAAGACTGGGCCTTTCG,s107_SI-DNA,0
2,ST1-AAAAGACTGGGCCTTTCG,s108_SI-DNA,997
3,ST1-AAAAGACTGGGCCTTTCG,s117_Cecum-DNA,13533
4,ST1-AAAAGACTGGGCCTTTCG,s118_Cecum-DNA,53880
...,...,...,...
189155,ST1-TTTTGTGCTATTGCGGTT,NTC_H3_NTC,0
189156,ST1-TTTTGTGCTATTGCGGTT,WATER_B3_WATER,0
189157,ST1-TTTTGTGCTATTGCGGTT,WATER_C6_WATER,0
189158,ST1-TTTTGTGCTATTGCGGTT,WATER_D2_WATER,0


In [13]:
df_stack.groupby('sample').agg(**{'quant95': ('clustered_molecules', lambda x: x.quantile(0.95)),
                                'quant97': ('clustered_molecules', lambda x: x.quantile(0.97)),
                                'quant99': ('clustered_molecules', lambda x: x.quantile(0.99))})

,quant95,quant97,quant99
sample,,,
NTC_H3_NTC,9.00,19.00,113.43
WATER_B3_WATER,0.00,1.00,1.00
WATER_C6_WATER,1.00,1.00,30.43
WATER_D2_WATER,2.00,4.00,12.00
WATER_F9_WATER,0.00,0.00,1.00
s107_SI-DNA,22.00,67.00,1705.27
s108_SI-DNA,1.00,1.00,7.00
s117_Cecum-DNA,133.00,200.00,467.00
s118_Cecum-DNA,120.00,160.00,360.00
